# LCProp LC

Use a fresh CPU runtime with the reviewed LCProp wheel and its Colab extra. Verify the wheel checksum and restart after installation before running the display cell. This notebook opens an independent controller. Only the interface Run button starts propagation.


## Install once in a fresh runtime

Upload the staged release wheel into `/content` using the Colab Files pane. Run the install cell, restart the session, then start at the verification cell. The install cell also repairs missing Jedi in Colab images where IPython requires it.

In [ ]:
import hashlib
from pathlib import Path
import subprocess
import sys

wheel = Path("/content/lcprop-0.5.0rc2-py3-none-any.whl")
expected_sha256 = "4389e3acc5f0b3c48e593abab6d61353a9c6df0c92258e539223326e6270a2cc"
if hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError("The uploaded wheel does not match the pinned release candidate.")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", str(wheel)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", f"lcprop[colab] @ {wheel.as_uri()}", "jedi"],
    check=True,
)
print("Restart the session and continue at Verify installation.")

## Verify installation

Run this cell after restarting. Then run the interface cell below.

In [ ]:
from importlib import metadata
import json
import subprocess
import sys

expected_sha256 = "4389e3acc5f0b3c48e593abab6d61353a9c6df0c92258e539223326e6270a2cc"
dist = metadata.distribution("lcprop")
origin = json.loads(dist.read_text("direct_url.json") or "{}")
installed_sha256 = origin.get("archive_info", {}).get("hashes", {}).get("sha256")
if dist.version != "0.5.0rc2" or installed_sha256 != expected_sha256:
    raise RuntimeError("Install the pinned wheel and restart this session first.")
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
print("Verified LCProp", dist.version, installed_sha256)

In [ ]:
from google.colab import output
from IPython import get_ipython
from IPython.display import clear_output
from lcprop.notebook import LCNotebook

if "lcprop_lc" not in globals():
    output.enable_custom_widget_manager()
    get_ipython().run_line_magic("matplotlib", "widget")
    lcprop_lc = LCNotebook()
elif lcprop_lc.session.disposed:
    raise RuntimeError("Restart after saving needed files to replace a disposed controller.")
clear_output(wait=True)
lcprop_lc.display_in_colab()
